## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cmocean
import matplotlib

plt.rcParams['figure.dpi'] = 300
matplotlib.rcParams["font.family"]      = "DejaVu Sans"
matplotlib.rcParams["figure.facecolor"] = "white"
matplotlib.rcParams["axes.facecolor"]   = "white"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Load Pretrained Models

In [ ]:
from models.unet.default import Unet

N_EULER = 48  # forward Euler steps used for optimization, sampling, and rendering


def load_model(n_circles: int, epoch: int = 90) -> torch.nn.Module:
    ckpt = f"../../outputs/fm_unet/n{n_circles}/checkpoints/epoch_{epoch:04d}.pt"
    model = Unet(ch=16, ch_mul=[1, 2, 4], att_channels=[0, 0, 1], groups=8, dropout=0.0).to(device)
    state = torch.load(ckpt, map_location=device, weights_only=True)
    new_state = {k.replace("module._orig_mod.", ""): v
                 for k, v in state.items() if k != "n_averaged"}
    model.load_state_dict(new_state)
    model.eval()
    print(f"  n={n_circles} loaded from {ckpt}")
    return model

models = {1: load_model(1)}


def euler_integrate(model, x0, n_steps=N_EULER):
    """Forward Euler integration of the velocity field from t=0 to t=1."""
    dt = 1.0 / n_steps
    x = x0
    for i in range(n_steps):
        t = torch.full((x.shape[0],), i * dt, device=x.device)
        x = x + dt * model(x, t)
    return x


In [ ]:
BLUR_SIGMA = 5.0   # σ=2 keeps enough frequencies for D-Flow to recover structure.

def make_blur(sigma: float, device):
    ks = int(6 * sigma + 1)
    ks = ks if ks % 2 == 1 else ks + 1
    half = ks // 2
    coords = torch.arange(ks, dtype=torch.float32) - half
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = (g / g.sum()).to(device)

    conv_x = torch.nn.Conv2d(1, 1, kernel_size=(1, ks), padding=(0, half), bias=False).to(device)
    conv_y = torch.nn.Conv2d(1, 1, kernel_size=(ks, 1), padding=(half, 0), bias=False).to(device)
    with torch.no_grad():
        conv_x.weight.copy_(g.view(1, 1, 1, ks))
        conv_y.weight.copy_(g.view(1, 1, ks, 1))
    for p in (*conv_x.parameters(), *conv_y.parameters()):
        p.requires_grad_(False)

    def H(x: torch.Tensor) -> torch.Tensor:
        return conv_y(conv_x(x))
    return H

H = make_blur(BLUR_SIGMA, device)

splits = torch.load("../../data/circles-64-n1.pt", weights_only=True)
sample = splits["test"][0].unsqueeze(0).to(device)
blurred = H(sample)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), facecolor="white",
                                gridspec_kw={"wspace": 0.05}, dpi=400)
for ax, img, title in [(ax1, sample, "Sharp"), (ax2, blurred, f"Blurred (σ={BLUR_SIGMA})")]:
    ax.imshow(img.squeeze().cpu().numpy(), cmap=cmocean.cm.dense_r, vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
plt.show()


$$\mathcal{L} = ||\text{H}(x_0) - y_{\text{obs}}||^2$$

In [ ]:
def mse_cost(x, y):
    return (x - y).pow(2).mean()

## Pre-D-Flow Sample Grid

Sanity-check the unconditional prior: a small grid of samples drawn from the
n=1 model with `N_EULER` forward Euler steps before any D-Flow optimisation.

In [ ]:
torch.manual_seed(0)
GRID_R, GRID_C = 4, 4
n_grid = GRID_R * GRID_C

with torch.no_grad():
    x0_grid = torch.randn(n_grid, 1, 64, 64, device=device)
    samples_grid = euler_integrate(models[1], x0_grid, n_steps=N_EULER).clamp(0, 1)

fig, axes = plt.subplots(GRID_R, GRID_C,
                         figsize=(GRID_C * 1.5, GRID_R * 1.5),
                         facecolor="white",
                         gridspec_kw={"hspace": 0.05, "wspace": 0.05})
for i, ax in enumerate(axes.flat):
    ax.imshow(samples_grid[i, 0].cpu().numpy(), cmap=cmocean.cm.dense_r,
              vmin=0, vmax=1, interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
fig.suptitle(f"Unconditional samples — n=1, {N_EULER} Euler steps",
             fontsize=10, y=0.92)
plt.show()


## D-Flow Optimisation (Adam)

## D-Flow: n = 1

Converges quickly — can use a larger learning rate.

In [ ]:
import time

torch.manual_seed(1234)
results = {}

# ── n=1 hyperparameters ──────────────────────────────────────────────────────
N1_EULER_OPT  = N_EULER
N1_EULER_REND = N_EULER
N1_LR         = 1e-2          # constant Adam step size (no schedule)
N1_BETAS      = (0.9, 0.999)
N1_STEPS      = 500_000
N1_STOP_LOSS  = 1e-8
LOG_EVERY     = 1000

model = models[1]
x0_gt = torch.randn(1, 1, 64, 64, device=device)

with torch.no_grad():
    y_gt  = euler_integrate(model, x0_gt, n_steps=N1_EULER_REND).clamp(0, 1)
    y_obs = H(y_gt)

x0 = torch.randn(1, 1, 64, 64, device=device, requires_grad=True)
optimizer = torch.optim.Adam([x0], lr=N1_LR, betas=N1_BETAS)

best_loss = float("inf")
best_x0   = x0.detach().clone()
best_step = -1

print("n=1:")
if torch.cuda.is_available():
    torch.cuda.synchronize()
t_start = time.perf_counter()
for step in range(N1_STEPS):
    optimizer.zero_grad()
    y_pred = euler_integrate(model, x0, n_steps=N1_EULER_OPT)
    loss = mse_cost(H(y_pred), y_obs)
    loss.backward()
    torch.nn.utils.clip_grad_norm_([x0], max_norm=1.0)
    optimizer.step()

    cur = loss.item()
    if cur < best_loss:
        best_loss = cur
        best_x0   = x0.detach().clone()
        best_step = step

    if step % LOG_EVERY == 0:
        grad_norm = x0.grad.norm().item() if x0.grad is not None else 0.0
        elapsed = time.perf_counter() - t_start
        print(f"  [step {step:6d}] loss={cur:.3e}  best={best_loss:.3e}  "
              f"|grad|={grad_norm:.3e}  lr={N1_LR:.3e}  elapsed={elapsed:7.2f}s")

    if cur < N1_STOP_LOSS:
        print(f"  [step {step:6d}] converged — loss={cur:.3e}")
        break

if torch.cuda.is_available():
    torch.cuda.synchronize()
total_time = time.perf_counter() - t_start
print(f"  best loss={best_loss:.6e} at step {best_step}")
print(f"  total elapsed: {total_time:.2f}s ({total_time/60:.2f} min)")

with torch.no_grad():
    y_hat = euler_integrate(model, best_x0, n_steps=N1_EULER_REND).clamp(0, 1)

results[1] = {
    "y_gt": y_gt.cpu(), "y_obs": y_obs.cpu(),
    "y_hat": y_hat.cpu(), "err": (y_hat - y_gt).cpu(),
    "best_loss": best_loss, "best_step": best_step,
    "elapsed_s": total_time,
}


## Results

In [ ]:
row_labels = [r"Sharp $y_{\mathrm{gt}}$", r"Blurred $y_{\mathrm{obs}}$",
              r"D-Flow $\hat{y}$", r"Error $\hat{y} - y_{\mathrm{gt}}$"]
nrows = len(row_labels)

fig, axes = plt.subplots(nrows, 1, figsize=(4, 16), facecolor="white",
                         gridspec_kw={"hspace": 0.12}, dpi=400)

n = 1
res = results[n]
elim = max(res["err"].abs().max().item(), 1e-8)
mse = res["err"].pow(2).mean().item()

for row, (img, cmap, vmin, vmax) in enumerate([
    (res["y_gt"],  cmocean.cm.dense_r, 0, 1),
    (res["y_obs"], cmocean.cm.dense_r, 0, 1),
    (res["y_hat"], cmocean.cm.dense_r, 0, 1),
    (res["err"],   cmocean.cm.balance, -elim, elim),
]):
    ax = axes[row]
    im = ax.imshow(img.squeeze().numpy(), cmap=cmap, vmin=vmin, vmax=vmax,
                   interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_ylabel(row_labels[row], fontsize=10, rotation=0, labelpad=70,
                  va="center", ha="left")

axes[0].set_title(f"n = {n}", fontsize=11, pad=6)
axes[2].set_xlabel(f"MSE = {mse:.3e}", fontsize=8, labelpad=4)

cbar = fig.colorbar(im, ax=axes[3], location="bottom",
                    fraction=0.06, pad=0.08, aspect=40)
cbar.ax.tick_params(labelsize=8)

fig.suptitle(rf"D-Flow deblurring (Adam) — $\sigma = {BLUR_SIGMA}$, "
             rf"{N_EULER} Euler steps", fontsize=13, y=0.98)
plt.show()
